## Spring Working Connections 2026
### Intro to Quantum Workshop Final Assessment

- Follow the steps provided below to complete your final assessment to earn the workshop Credly badge.
- **You may use any code examples provided in the workshop notebooks as necessary to implement your solution.**
- Use the code block provided to enter and execute your code. The setup block is provided before the template block to support import operations in your code.
- Once completed, you can either save your notebook to your forked repo in GitHub and provide me with the URL, or download the notebook (use the Colab menu bar: File/Download/Download .ipynb) and email the download notebook to me so I can view and execute your code.


### Rubric
- Your work will be scored based on completion of the following tasks.
- **Partial credit will be given! (don't leave anything blank)**
- A minimum score of **80%** is required to earn the badge.
<br>

- Initialization + State Table (20%): correct 3-qubit superposition and pre-oracle state table
- Phase Oracle (20%): correct two-target oracle and post-oracle state table
- Grover Operator (20%): one correct Grover iteration with visible amplitude change
- Measurement (20%): runs with shots = 1000; marked states appear more frequently
- Complete, Working Program (20%): runs without errors; all required outputs shown

## Final Assessment: Grover Search with Two Targets
### Objective
- Use a quantum circuit to identify target values from a set of possibilities by:
  - encoding the problem as a phase oracle
  - applying the Grover operator
  - analyzing state tables and measurement results
- Problem Description
  - You are given a “database” of all 3-qubit outcomes (000, 001, 010, 011, 100, 101, 110, 111)
  - Your task is to mark two target values and observe how the circuit behavior changes.
  - Use k = 3 and k = 6 as the marked outcomes and start with a uniform superposition
- Steps
  0. Run the setup code block
  1. Define the target predicate for marked outcomes (k = 3, 6)
  2. Define the phase oracle to flip amplitudes of marked states
  3. Define the diffusion operator (inversion about the mean)
  4. Initialize a 3-qubit circuit and apply Hadamard gates to create a uniform superposition
  5. Run the circuit and display the initial state table
  6. Copy the original state to preserve it for the Grover iteration
  7. Apply the oracle to the current state and display the result
  8. Apply one Grover iteration (oracle + diffusion) using the saved original state
  9. Measure the final state over multiple shots and display counts
###Implementation Notes
- Use the provided object-oriented framework and classical functions for the oracle and grover implementation
  - QuantumRegister
  - QuantumCircuit
  - oracle(...)
  - run()
- Use:
  - print_state_table(...) for output
  - the provided measure(...) function for sampling
### Deliverables
1. Code
  - Submit a complete working program in the provided code block in the assessment notebook (refresh your repo fork, then access the notebook in the refreshed content).
2. Output
  - State table before oracle
  - State table after oracle
  - State table after one Grover iteration
  - Measurement results (shots = 1000)



In [1]:
# ---- setup for clone/repo imports ----
import os
import sys
import subprocess
import importlib

REPO_URL = "https://github.com/learnqc/code.git"
REPO_DIR = "/content/code"
SRC_DIR = f"{REPO_DIR}/src"

# Install required pip package(s)
subprocess.run(
    ["pip", "install", "-q", "sty"],
    check=True
)

# Clone repo if needed
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

# Make src importable
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Clear any stale imports
importlib.invalidate_caches()

print("Setup complete")

Setup complete


In [2]:
from ch03.util import *
from ch05.sim_circuit import *

# classical phase oracle (acts directly on the state)
def classical_phase_oracle(state, predicate):
    for item in range(len(state)):
        if predicate(item):
            state[item] *= -1

# predicate function: returns True if k == 3 or 6
def is_3or6(k):
    return k in [3, 6]

predicate = is_3or6

# create a 3-qubit register and circuit
q = QuantumRegister(3)
qc = QuantumCircuit(q)

n = len(q)

# initialize to a uniform superposition
qc.initialize([1 / sqrt(2**n) for _ in range(2**n)])

print(f"\nGood outcomes: {[k for k in range(2**n) if predicate(k)]}")

print("Before oracle")
state = qc.run()
print_state_table(state)

# apply the classical phase oracle directly to the state
classical_phase_oracle(state, predicate)

print("After oracle")
print_state_table(state)






Good outcomes: [3, 6]
Before oracle

Outcome  Binary  Amplitude           Direction  Magnitude  Amplitude Bar             Probability
------------------------------------------------------------------------------------------------
0        000     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
1        001     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
2        010     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
3        011     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
4        100     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
5        101     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
6        110     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
7        111     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 

After oracle

Outcome  Binary  

In [3]:
from math import sqrt, sin, cos, asin
from ch03.util import *
from ch05.sim_circuit import *

# inner product <a|b>
def inner(a, b):
    total = 0
    for k in range(len(a)):
        total += a[k].conjugate() * b[k]
    return total

# classical phase oracle from Day 5
def oracle(state, predicate):
    for k in range(len(state)):
        if predicate(k):
            state[k] *= -1


# inversion operator
def inversion(original, current):
    proj = inner(original, current)
    for k in range(len(current)):
        current[k] = 2 * proj * original[k] - current[k]

# compare floating-point values
def is_close(a, b, tol=1e-9):
    return abs(a - b) < tol

# classical Grover iteration
def classical_grover(state, predicate, iterations):
    s = state.copy()
    items = [k for k in range(len(state)) if predicate(k)]

    # probability of measuring a good outcome in the initial state
    p = sum(abs(s[k])**2 for k in items)
    theta = asin(sqrt(p))

    # before any Grover iterations, <s|state> should be 1
    assert is_close(inner(s, state), 1)

    for it in range(1, iterations + 1):
        oracle(state, predicate)
        inversion(s, state)

        # check the angle relationship after each Grover iteration
        assert is_close(inner(s, state), cos(2 * it * theta))

        # check the probability of good outcomes after each iteration
        p = sum(abs(state[k])**2 for k in items)
        assert is_close(p, sin((2 * it + 1) * theta)**2)

# expected amplitude for a good outcome in the uniform case
def target_amplitude_uniform(n, l, j):
    theta = asin(sqrt(l / 2**n))
    return sin((2 * j + 1) * theta) / sqrt(l)

# predicate function: returns True if k == 3 or 6
def is_3or6(k):
    return k in [3, 6]

predicate = is_3or6


## common setup
n = 3
#items = [3]
#predicate = lambda i: i in items


# create equal superposition over 3 qubits
# Step 1: create a 3-qubit circuit
q = QuantumRegister(n)
qc = QuantumCircuit(q)

# Step 2: apply H to all qubits for a uniform superposition
for i in range(n):
    qc.h(q[i])

# Step 3: use qc.run() to get the initial state
initial_state = qc.run()

print(f"\nGood outcomes: {[k for k in range(2**n) if predicate(k)]}")

s = state.copy()

print("Original state before oracle")
print_state_table(state)

# apply the classical phase oracle directly to the state
oracle(state, predicate)

print("After oracle (good outcome phase-flipped)")
print_state_table(state)

# apply inversion operator
inversion(s, state)

print("After inversion")
print_state_table(state)




# --------
# 1 iteration
# --------

print("Initial state (before Grover, j = 0)")
print_state_table(state)

classical_grover(state, predicate, iterations=1)

print("After 1 Grover iteration")
print_state_table(state)

#assert is_close(state[items[0]], target_amplitude_uniform(3, 1, 1))







Good outcomes: [3, 6]
Original state before oracle

Outcome  Binary  Amplitude           Direction  Magnitude  Amplitude Bar             Probability
------------------------------------------------------------------------------------------------
0        000     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
1        001     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
2        010     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
3        011    -0.3536 + i0.0000     180.00°   0.3536     ████████                  0.125 
4        100     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
5        101     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
6        110    -0.3536 + i0.0000     180.00°   0.3536     ████████                  0.125 
7        111     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 

After oracle (go

In [9]:
#trying again at 4:24 PM

from math import sqrt, sin, cos, asin
from ch03.util import *
from ch05.sim_circuit import *

# inner product <a|b>
def inner(a, b):
    total = 0
    for k in range(len(a)):
        total += a[k].conjugate() * b[k]
    return total

# classical phase oracle
def oracle(state, predicate):
    for item in range(len(state)):
        if predicate(item):
            state[item] *= -1

# inversion operator
def inversion(original, current):
    proj = inner(original, current)
    for k in range(len(current)):
        current[k] = 2 * proj * original[k] - current[k]

# compare floating-point values
def is_close(a, b, tol=1e-9):
    return abs(a - b) < tol

# Grover iteration
def grover(state, predicate, iterations):
  # original state is a reference point needed for the
  # reflection so we save it before applying the oracle
    s = state.copy()
    print("Initial state in Grover")
    print_state_table(state)
    items = [k for k in range(len(state)) if predicate(k)]

    # probability of measuring a good outcome in the initial state
    p = sum(abs(s[k])**2 for k in items)
    theta = asin(sqrt(p))

    # before any Grover iterations, <s|state> should be 1
    assert is_close(inner(s, state), 1)

    for it in range(1, iterations + 1):
        #print("before oracle")
        #print_state_table(state)
        oracle(state, predicate)
        print("after oracle")
        print_state_table(state)
        inversion(s, state)

        # check the angle relationship after each Grover iteration
        assert is_close(inner(s, state), cos(2 * it * theta))

        # check the probability of good outcomes after each iteration
        p = sum(abs(state[k])**2 for k in items)
        #assert is_close(p, sin((2 * it + 1) * theta)**2)

# expected amplitude for a good outcome in the uniform case
def target_amplitude_uniform(n, l, j):
    theta = asin(sqrt(l / 2**n))
    return sin((2 * j + 1) * theta) / sqrt(l)

# mark outcomes 3 & 6 as good outcomes
# predicate function: returns True if k == 3 or 6
def is_3or6(k):
    return k in [3, 6]

predicate = is_3or6

# Step 1: create a 3-qubit circuit
q = QuantumRegister(n)
qc = QuantumCircuit(q)

# Step 2: apply H to all qubits for a uniform superposition
for i in range(n):
    qc.h(q[i])

# Step 3: use qc.run() to get the initial state
initial_state = qc.run()

state = initial_state.copy()
print(f"\nGood outcomes: {[k for k in range(2**n) if predicate(k)]}")

grover(state, predicate, iterations=1)

print("After 1 Grover iteration")
print_state_table(state)

#assert is_close(state[items[0]], target_amplitude_uniform(3, 1, 1))


Good outcomes: [3, 6]
Initial state in Grover

Outcome  Binary  Amplitude           Direction  Magnitude  Amplitude Bar             Probability
------------------------------------------------------------------------------------------------
0        000     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
1        001     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
2        010     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
3        011     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
4        100     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
5        101     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
6        110     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
7        111     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 

after oracle

Outcome